# 06 — Eigenvalue Verification: GUE Spacing Analysis

> *Large-N results were generated with the proprietary Isomorphic Engine; this notebook reproduces the analysis and figures from the released data.*

This notebook loads two verification datasets:
1. **200-zero KS test** (`eigenvalue_verification_200.json`) — Kolmogorov–Smirnov comparison of Riemann zero spacings vs GUE predictions
2. **1000-zero GPU run** (`riemann_gpu_tf32_1000.json`) — RTX 5070 Ti TF32 computation at 100-digit precision

We reproduce:
- Spacing distribution histograms
- KS test statistics and p-values
- Pair correlation function $R_2(r)$ vs GUE prediction
- 2-sigma agreement bands

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': '#0A2540',
    'axes.facecolor': '#0A2540',
    'axes.edgecolor': '#D4AF37',
    'axes.labelcolor': 'white',
    'text.color': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'figure.figsize': (12, 5),
    'font.size': 12
})

## 1. Load verification data

In [ ]:
with open('../data/eigenvalue-verification/eigenvalue_verification_200.json') as f:
    ev200 = json.load(f)

with open('../data/riemann-zeros/riemann_gpu_tf32_1000.json') as f:
    gpu1000 = json.load(f)

print(f"200-zero dataset: {ev200['n_zeros']} zeros, KS p-value = {ev200['statistics']['ks_p_value']:.4f}")
print(f"1000-zero dataset: GPU = {gpu1000['computation']['gpu']}")
print(f"  Precision: {gpu1000['computation']['precision_digits']} digits")
print(f"  Pair correlation MSE vs GUE: {gpu1000['pair_correlation']['mse_vs_gue_r2']:.4f}")

## 2. 200-zero spacing comparison

In [ ]:
# Extract table data
table = ev200['table']
k_vals = [row['k'] for row in table]
s_zeta = [row['s_zeta'] for row in table]
s_gue = [row['s_gue'] for row in table]
within_2sigma = [row['within_2sigma'] for row in table]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter plot of spacings
ax = axes[0]
colors = ['#D4AF37' if w else '#FF4444' for w in within_2sigma]
ax.scatter(s_gue, s_zeta, c=colors, s=15, alpha=0.8, zorder=3)
lims = [min(min(s_zeta), min(s_gue)) * 0.9, max(max(s_zeta), max(s_gue)) * 1.1]
ax.plot(lims, lims, '--', color='white', alpha=0.5, lw=1)
ax.set_xlabel('GUE predicted spacing')
ax.set_ylabel('Riemann zero spacing')
ax.set_title(f'200-Zero GUE Comparison ({ev200["statistics"]["match_percent"]}% within 2σ)')

# Right: KS test visualization
ax = axes[1]
s_zeta_sorted = np.sort(s_zeta)
s_gue_sorted = np.sort(s_gue)
n = len(s_zeta_sorted)
ecdf_zeta = np.arange(1, n+1) / n
ecdf_gue = np.arange(1, n+1) / n
ax.step(s_zeta_sorted, ecdf_zeta, color='#D4AF37', lw=2, label='Riemann zeros')
ax.step(s_gue_sorted, ecdf_gue, color='#66B2FF', lw=2, label='GUE prediction', linestyle='--')
ax.set_xlabel('Spacing')
ax.set_ylabel('CDF')
ax.set_title(f'KS Test: D = {ev200["statistics"]["ks_statistic"]:.4f}, p = {ev200["statistics"]["ks_p_value"]:.4f}')
ax.legend(facecolor='#0A2540', edgecolor='#D4AF37')

plt.tight_layout()
plt.savefig('../figures/eigenvalue_200_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 1000-zero spacing distribution

In [ ]:
# Spacing statistics from GPU run
sp = gpu1000['spacings']
print(f"Spacing statistics (n={sp['n']} gaps):")
print(f"  Mean:     {sp['mean']:.6f}")
print(f"  Std:      {sp['std']:.4f}")
print(f"  Skewness: {sp['skew']:.3f}")
print(f"  Kurtosis: {sp['kurtosis']:.3f}")
print(f"  Range:    [{sp['min']:.4f}, {sp['max']:.4f}]")
print()

# GUE comparison
gue = gpu1000['gue_comparison']
print(f"GUE comparison:")
print(f"  KS statistic:  {gue['ks_statistic']:.4f}")
print(f"  KS p-value:    {gue['ks_p_value']}")
print(f"  AD statistic:  {gue['ad_statistic']:.3f}")
print()

# Pair correlation
pc = gpu1000['pair_correlation']
print(f"Pair correlation R₂(r):")
print(f"  MSE vs GUE: {pc['mse_vs_gue_r2']:.4f}")

## 4. First 20 zeros — high-precision values

In [ ]:
first_20 = gpu1000['zeros']['first_20']
print(f"First 20 non-trivial Riemann zeros (imaginary parts):")
print(f"{'k':>4}  {'γ_k':>20}")
print('-' * 26)
for i, z in enumerate(first_20, 1):
    print(f"{i:>4}  {z:>20.12f}")

## 5. Summary

| Dataset | Test | Result |
|---------|------|--------|
| 200 zeros | KS test | p = 0.0512 (marginal, consistent with GUE) |
| 200 zeros | 2σ agreement | 91.2% of spacings within band |
| 200 zeros | Spearman ρ | 1.000 (perfect rank correlation) |
| 1000 zeros | Pair correlation MSE | 0.752 vs GUE R₂ |
| 1000 zeros | Computation | RTX 5070 Ti, 100-digit precision |